# Notebook 2 — Base Case Power Flow

**Course: Cross-Corridor Capacity Analysis with pandapower (Svedala grid)**

Before any contingency or capacity analysis, we verify the **base case** — the
snapshot as-delivered, with all elements in service. Three checks:

1. The AC power flow converges.
2. No bus voltage is outside the band [0.95, 1.05] p.u.
3. No line/transformer is overloaded.

## Learning objectives

- Run an AC power flow with `pp.runpp()`.
- Read result tables (`net.res_bus`, `net.res_line`, `net.res_trafo`).
- Build a small reusable helper `check_limits()` that returns the violations.
- Note that for a real CIM-imported model, the base case is *not guaranteed* to
  be feasible — and that's a real-world finding worth examining.

## 2.1  Imports and load the grid

In [ ]:
import pandapower as pp
import pandas as pd
import matplotlib.pyplot as plt

net = pp.from_json('data/svedala.json')
print(net)

## 2.2  Run the power flow

If the algorithm fails to converge it raises `LoadflowNotConverged`. We wrap
the call in `try/except` because in the analyses to come, non-convergence is
common and we don't want it to crash the whole study.

In [ ]:
try:
    pp.runpp(net)
    print('✓ Power flow converged')
except pp.LoadflowNotConverged:
    print('✗ Power flow did NOT converge')

## 2.3  Bus voltages

In [ ]:
V_MIN, V_MAX = 0.95, 1.05

vb = net.res_bus.copy()
vb['name'] = net.bus.name.values
vb['zone'] = net.bus.zone.values
vb['vn_kv'] = net.bus.vn_kv.values

violators = vb[(vb.vm_pu < V_MIN) | (vb.vm_pu > V_MAX)]
print(f'{len(violators)} bus voltage violation(s) in base case')
violators[['name', 'zone', 'vn_kv', 'vm_pu']].round(3)

## 2.4  Line loadings

In [ ]:
LINE_LIMIT_PCT = 100.0

rl = net.res_line.copy()
rl['name']     = net.line.name.values
rl['from_bus'] = net.line.from_bus.values
rl['to_bus']   = net.line.to_bus.values

# Top-10 most-loaded lines
rl.sort_values('loading_percent', ascending=False).head(10)[
    ['name', 'from_bus', 'to_bus', 'loading_percent']].round(1)

In [ ]:
line_violators = rl[rl.loading_percent > LINE_LIMIT_PCT]
print(f'{len(line_violators)} line overload(s) in base case')
line_violators[['name', 'from_bus', 'to_bus', 'loading_percent']].round(1)

## 2.5  Transformer loadings

In [ ]:
TRAFO_LIMIT_PCT = 100.0

rt = net.res_trafo.copy()
rt['name'] = net.trafo.name.values
trafo_violators = rt[rt.loading_percent > TRAFO_LIMIT_PCT]
print(f'{len(trafo_violators)} transformer overload(s) in base case')
trafo_violators[['name', 'loading_percent']].round(1)

## 2.6  Reusable `check_limits()` helper

In [ ]:
def check_limits(net, v_min=0.95, v_max=1.05,
                 line_limit=100.0, trafo_limit=100.0):
    """Return a dict with indices of all violating elements."""
    return {
        'voltage_low':    net.res_bus[net.res_bus.vm_pu < v_min].index.tolist(),
        'voltage_high':   net.res_bus[net.res_bus.vm_pu > v_max].index.tolist(),
        'line_overload':  net.res_line[net.res_line.loading_percent > line_limit].index.tolist(),
        'trafo_overload': net.res_trafo[net.res_trafo.loading_percent > trafo_limit].index.tolist(),
    }

v = check_limits(net)
for k, idx in v.items():
    print(f'  {k:<16}  {len(idx)}')

> ⚠️ **Reality check.** A CIM-imported snapshot from a real grid model is *not*
> guaranteed to be feasible against simple 0.95–1.05 / 100 % limits. Real
> operating grids run with tap changers, switched shunts and OPF redispatch
> — none of which are active here. If your base case shows violations,
> document them and proceed: in the contingency / capacity analyses we will
> measure changes *relative* to this state, so the result is still meaningful.

## 2.7  Loading distribution by voltage level

A quick visual: which voltage level is most stressed in the base case?

In [ ]:
rl['vn_kv'] = [net.bus.at[i, 'vn_kv'] for i in rl.from_bus]
fig, ax = plt.subplots(figsize=(8, 4))
for vn, sub in rl.groupby('vn_kv'):
    ax.hist(sub.loading_percent, bins=20, alpha=0.6, label=f'{vn:.0f} kV')
ax.axvline(100, color='red', linestyle='--', label='100 %')
ax.set_xlabel('Loading [%]'); ax.set_ylabel('Number of lines')
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_title('Distribution of line loadings in the base case')
plt.tight_layout(); plt.show()

## 2.8  Save the base-case net

In [ ]:
pp.to_json(net, 'data/svedala_base.json')
print('Saved data/svedala_base.json (base case after PF)')

## 2.9  Exercises

1. List the three buses with the **lowest** voltage and the three with the
   **highest**. Which zones are they in? Does this match your intuition that
   long-distance transfer typically depresses voltage at the receiving end?
2. Which transformers (by name) are the most loaded? Are any of them tap
   changers (`tap_changer_type` is not NaN)? If yes, real-time tap action
   would mitigate; in this static study the loading is what it is.
3. Sum the active power injected by the slack generator. Is it small (good —
   the dispatch is roughly balanced) or large (the dispatch is far off, and
   the slack is doing significant work)?

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3


---

✅ **Checkpoint reached.** Base case verified, `check_limits()` ready.

Continue to **Notebook 3** — `03_corridor_definition.ipynb`.